# Lab 2: Containerization & Packaging with Docker

## Learning Objectives
By the end of this lab, you will:
- Write a two-stage Dockerfile that produces a lean, secure AI API image
- Configure `docker-compose.yml` to orchestrate the API, Qdrant, and Prometheus
- Apply security best practices: non-root user, HEALTHCHECK, `.dockerignore`
- Explain why each `.dockerignore` rule matters for security or performance

## Overview
AI applications suffer from dependency hell: conflicting CUDA versions, PyTorch build tags, and native libraries that differ across machines. Docker solves this by packaging your app and all its dependencies into an immutable image that runs identically everywhere.

## What You'll Build
- A multi-stage `Dockerfile` (builder stage + lean runtime stage)
- A `docker-compose.yml` orchestrating 3 services: API, Qdrant, Prometheus
- A `.dockerignore` file with explanations for each exclusion rule

In [ ]:
import os
print("Environment check:")
os.system("docker --version")
os.system("docker compose version")

---
## Part 1: Writing the Multi-Stage Dockerfile (30 min)

A multi-stage build uses two `FROM` instructions:
- **Stage 1 (builder):** Install compilers, build Python wheels, create virtualenv
- **Stage 2 (runtime):** Copy only the virtualenv -- no build tools in final image

This typically reduces image size from ~2.3 GB to ~450 MB.

### Exercise 1.1: Write the Dockerfile

Fill in each `???` in the template below using the hints provided. Save the result as `Dockerfile`.

In [ ]:
dockerfile_template = '''# ── Stage 1: Builder ─────────────────────────────────────────────
FROM python:3.11-slim AS builder

# TODO 1: Install build-time system dependencies
# Hint: you need gcc, g++, and curl for compiling Python extensions
# RUN apt-get update && apt-get install -y --no-install-recommends \
#     gcc g++ curl \
#     && rm -rf /var/lib/apt/lists/*

WORKDIR /app
COPY requirements.txt .

# TODO 2: Create an isolated virtual environment and install dependencies
# Hint: create venv at /opt/venv, then activate it by setting PATH
# RUN python -m venv /opt/venv
# ENV PATH="/opt/venv/bin:$PATH"
# RUN pip install --no-cache-dir --upgrade pip \
#  && pip install --no-cache-dir -r requirements.txt

# ── Stage 2: Runtime ─────────────────────────────────────────────
FROM python:3.11-slim AS runtime

# TODO 3: Install ONLY runtime dependencies (no gcc/g++ compiler)
# Hint: you only need curl (for healthcheck) and libglib2.0-0 (for sentence-transformers)
# RUN apt-get update && apt-get install -y --no-install-recommends \
#     curl libglib2.0-0 \
#     && rm -rf /var/lib/apt/lists/*

# TODO 4: Copy the virtual environment FROM the builder stage
# Hint: use COPY --from=builder <source> <dest> to cross-stage copy /opt/venv
# COPY --from=builder /opt/venv /opt/venv
# ENV PATH="/opt/venv/bin:$PATH"

# TODO 5: Create a non-root user for security
# Hint: useradd --create-home --shell /bin/bash appuser, then switch USER and WORKDIR
# RUN useradd --create-home --shell /bin/bash appuser
# USER appuser
# WORKDIR /home/appuser/app

# TODO 6: Copy application source with correct ownership
# Hint: COPY --chown=appuser:appuser ./src ./src
# COPY --chown=appuser:appuser ./src ./src

EXPOSE 8000

# TODO 7: Add a HEALTHCHECK for the /health endpoint
# Hint: --interval=30s --timeout=3s --start-period=10s --retries=3
#       CMD curl -f http://localhost:8000/health || exit 1
# HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 \
#     CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_template)
print("Dockerfile written -- now fill in the TODOs (uncomment and replace ??? with real values)")
print()
print(dockerfile_template)

In [ ]:
# Structural validation: check your Dockerfile has the key constructs
with open('Dockerfile') as f:
    content = f.read()

checks = [
    ("Two FROM instructions",          content.count("FROM python:3.11-slim") >= 2),
    ("Stage 1 named 'builder'",        "AS builder" in content),
    ("Stage 2 named 'runtime'",        "AS runtime" in content),
    ("gcc/g++ in builder",             "gcc" in content and "g++" in content),
    ("venv created at /opt/venv",      "/opt/venv" in content),
    ("Cross-stage COPY --from=builder","COPY --from=builder" in content),
    ("No gcc in runtime stage",        "gcc" not in content.split("AS runtime")[1] if "AS runtime" in content else False),
    ("Non-root user created",          "useradd" in content),
    ("USER appuser set",               "USER appuser" in content),
    ("HEALTHCHECK defined",            "HEALTHCHECK" in content),
    ("curl in HEALTHCHECK",            "curl -f http://localhost:8000/health" in content),
]

print("Dockerfile Validation Checklist")
print("=" * 55)
all_pass = True
for label, result in checks:
    status = "PASS" if result else "TODO"
    if not result:
        all_pass = False
    print(f"  [{status}] {label}")

print()
if all_pass:
    print("All checks passed! Your Dockerfile looks correct.")
else:
    print("Fix the failing checks, then re-run this cell.")

In [ ]:
# Key instruction explanations (reference)
explanations = [
    ("FROM python:3.11-slim AS builder",
     "Slim base: ~130MB vs 1GB for full Python. AS builder names this stage."),
    ("RUN python -m venv /opt/venv",
     "Isolated venv so COPY --from=builder copies only packages, not build tools."),
    ("COPY --from=builder /opt/venv /opt/venv",
     "Cross-stage copy: pulls compiled packages into clean runtime stage."),
    ("RUN useradd --create-home appuser",
     "Non-root user: limits blast radius if app is exploited."),
    ("HEALTHCHECK CMD curl -f http://localhost:8000/health",
     "Orchestrators (ECS, Kubernetes) use this to replace unhealthy containers."),
    ("COPY --chown=appuser:appuser ./src ./src",
     "Files owned by appuser -- avoids root-owned files when running as appuser."),
]

print("Dockerfile instruction explanations:")
for instr, explain in explanations:
    print(f"  {instr}")
    print(f"    -> {explain}")
    print()

---
## Part 2: Writing docker-compose.yml (40 min)

Docker Compose orchestrates multiple services as a single unit. Fill in the `???` placeholders in the template below.

### Exercise 2.1: Write `docker-compose.yml`

Use the service contract table from the slides:
| Service | Image/Build | Key Config |
|---|---|---|
| `api` | `build: .` | `env_file: .env`, `depends_on: qdrant (service_healthy)` |
| `qdrant` | `qdrant/qdrant:v1.9.0` | `volumes: qdrant_data:/qdrant/storage` |
| `prometheus` | `prom/prometheus:v2.51.0` | `volumes: ./monitoring/prometheus.yml:ro` |

In [ ]:
compose_template = '''version: "3.9"

services:
  api:
    build:
      context: .
      # TODO 1: Specify the Dockerfile name
      # Hint: the file is named "Dockerfile"
      dockerfile: ???
    ports:
      # TODO 2: Map host port 8000 to container port 8000
      # Hint: "HOST:CONTAINER"
      - "???:???"
    # TODO 3: Load environment variables from the .env file
    # Hint: use env_file: with a list containing ".env"
    env_file:
      - ???
    environment:
      # TODO 4: Pass Qdrant connection details
      # Hint: service name is "qdrant", port is 6333
      - QDRANT_HOST=???
      - QDRANT_PORT=???
    depends_on:
      qdrant:
        # TODO 5: Use service_healthy so api waits for qdrant's healthcheck to pass
        # Hint: the condition value is "service_healthy"
        condition: ???
    networks:
      - ai-net
    restart: unless-stopped

  qdrant:
    image: qdrant/qdrant:v1.9.0
    ports:
      - "6333:6333"
    volumes:
      # TODO 6: Mount a named volume for persistent Qdrant storage
      # Hint: named volume "qdrant_data" -> container path "/qdrant/storage"
      - ???:/qdrant/storage
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:6333/healthz"]
      interval: 30s
      timeout: 5s
      retries: 3
    networks:
      - ai-net

  prometheus:
    image: prom/prometheus:v2.51.0
    ports:
      - "9090:9090"
    volumes:
      # TODO 7: Mount prometheus config read-only
      # Hint: host path "./monitoring/prometheus.yml" -> container "/etc/prometheus/prometheus.yml"
      #       Add ":ro" at the end to make it read-only
      - ???:/etc/prometheus/prometheus.yml:ro
    networks:
      - ai-net

# TODO 8: Declare the named volume used by qdrant
# Hint: volumes section at root level, just "qdrant_data:" with no value
volumes:
  ???:

networks:
  ai-net:
    driver: bridge
'''

with open('docker-compose.yml', 'w') as f:
    f.write(compose_template)
print("docker-compose.yml written -- fill in the ??? placeholders")
print()
print(compose_template)

In [ ]:
# Structural validation: check your docker-compose.yml
import re

with open('docker-compose.yml') as f:
    compose = f.read()

# Check for unfilled placeholders
unfilled = compose.count("???")

checks = [
    ("No unfilled ??? placeholders",           unfilled == 0),
    ("Services: api defined",                  "  api:" in compose),
    ("Services: qdrant defined",               "  qdrant:" in compose),
    ("Services: prometheus defined",           "  prometheus:" in compose),
    ("api uses build: context",                "build:" in compose and "context: ." in compose),
    ("api has env_file: .env",                 "env_file:" in compose and ".env" in compose),
    ("depends_on uses service_healthy",        "service_healthy" in compose),
    ("qdrant volume mapped",                   "qdrant_data:/qdrant/storage" in compose),
    ("prometheus config mounted :ro",          "prometheus.yml" in compose and ":ro" in compose),
    ("Named volume declared at root level",    "^volumes:" in compose or "
volumes:" in compose),
    ("Network ai-net defined",                 "ai-net:" in compose),
]

print("docker-compose.yml Validation Checklist")
print("=" * 55)
all_pass = True
for label, result in checks:
    status = "PASS" if result else "TODO"
    if not result:
        all_pass = False
    print(f"  [{status}] {label}")

if unfilled > 0:
    print(f"\n  Found {unfilled} unfilled '???' placeholder(s) -- replace them all!")

print()
if all_pass:
    print("All checks passed! Run 'docker compose config' to verify syntax.")
else:
    print("Fix the failing checks, then re-run this cell.")

---
## Bonus: Write and Explain .dockerignore

Without `.dockerignore`, every `COPY . .` sends your entire repository context to Docker, including `.env` files with API keys, virtual environments (hundreds of MB), and git history.

In [ ]:
dockerignore_rules = [
    ("__pycache__/",       "Python bytecode -- regenerated at runtime, wastes 50-200 MB per layer"),
    ("*.py[cod]",          "Compiled Python files -- stale bytecode can cause subtle bugs"),
    (".venv/",             "Local venv -- image has its own venv from pip install step"),
    ("venv/",              "Alternative venv name -- also excluded"),
    (".env",               "CRITICAL: API keys/secrets -- baking into image is a security incident"),
    ("*.pem",              "TLS/SSH private keys -- must never be inside container images"),
    (".git/",              "Git history -- can be 100 MB+ and exposes commit diffs"),
    ("tests/",             "Test files -- not needed in production, increases image surface area"),
    ("notebooks/",         "Jupyter notebooks -- large, irrelevant to runtime"),
    ("*.ipynb",            "Notebook files -- also excluded explicitly for safety"),
    ("models/*.bin",       "Model weights can be 1-7 GB -- mount as Docker volumes instead"),
    (".pytest_cache/",     "Test artifacts -- add noise without benefit"),
]

content = "# .dockerignore\n"
for rule, reason in dockerignore_rules:
    content += f"# {reason}\n{rule}\n\n"

with open('.dockerignore', 'w') as f:
    f.write(content)

print(".dockerignore written")
print()
print(f"{'Rule':<25} {'Why it matters'}")
print("-" * 80)
for rule, reason in dockerignore_rules:
    print(f"{rule:<25} {reason}")

---
## Reflection Questions

1. **Multi-stage size:** What is the approximate size difference between a single-stage and multi-stage build for this app? Why does it matter for pull time in CI/CD?

2. **Non-root user:** If your app has a remote code execution vulnerability and runs as root, what can an attacker do that they cannot do if running as `appuser`?

3. **depends_on condition:** Why do we use `condition: service_healthy` instead of just `depends_on: qdrant`? What problem does the health check condition solve?

In [ ]:
# Your answers here:
# 1.

# 2.

# 3.
